# Example 4: Batch Retrieval — Insulin Across Two Species


Fetch human INS (NM_000207) and mouse Ins2 (NM_008387) in a single Entrez call, and save them.


In [1]:
# Fetch human INS and mouse Ins2 mRNA records in one call, then save them
import csv
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)     # this Example creates the folders it needs.
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"  # NCBI requires a contact email
accessions = "NM_000207,NM_008387"  # Human INS, Mouse Ins2
handle = Entrez.efetch(db="nucleotide", id=accessions, rettype="gb", retmode="text")
records = list(SeqIO.parse(handle, "genbank"))
handle.close()

# Keep both raw records in one GenBank file
SeqIO.write(records, "data/insulin_INS_Ins2.gb", "genbank")

# Report each record's organism and coding-sequence length
rows = []
for rec in records:
    for feature in rec.features:
        if feature.type == "CDS":
            cds_len = len(feature.extract(rec.seq))
            print(f"{rec.id} ({rec.annotations['organism']}): CDS = {cds_len} bp")
            rows.append((rec.id, rec.annotations["organism"], cds_len))
            break

# Save the comparison as CSV
with open("results/example4_cds_lengths.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["accession", "organism", "cds_length_bp"])
    writer.writerows(rows)
print("Saved: data/insulin_INS_Ins2.gb, results/example4_cds_lengths.csv")

NM_000207.3 (Homo sapiens): CDS = 333 bp
NM_008387.5 (Mus musculus): CDS = 333 bp
Saved: data/insulin_INS_Ins2.gb, results/example4_cds_lengths.csv
